# Module 11 · Putting It Together: A Data Analysis in Pure Python

*BUSN 3030E — Python Programming Skills for Business Analytics I*

**Learning objectives.** After this module you will be able to

- read CSV and JSON data from the web into lists of dictionaries;
- clean raw values (currency symbols, missing values) with functions;
- answer analytical questions by combining loops, comprehensions, and dictionaries;
- present results in a formatted report.

This module uses only the tools from Modules 3–10, with no add-on libraries. Doing an analysis "by hand" once shows exactly what libraries such as `pandas` (Module 12 onward) automate for us.

## 1 · Preparation

### 1.1 Records as dictionaries

Tabular data (a spreadsheet, a database table, a CSV file) is a collection of **records** that share the same **fields**. In pure Python, a natural representation is a **list of dictionaries**: one dictionary per record, mapping field names to values.

```python
data = [
    {"Company": "Apple",  "Country": "USA",   "MarketCap": "$547"},
    {"Company": "Google", "Country": "USA",   "MarketCap": "$510"},
    ...
]
```

`data[0]` is the first record, `data[0]["Country"]` one of its fields, and `[r["Country"] for r in data]` a whole column.

### 1.2 Reading CSV files

A **CSV** (comma-separated values) file stores one record per line, with fields separated by commas and a header line with the field names. Python's built-in `csv` module parses it; `csv.DictReader` produces one dictionary per line.

In [1]:
import csv
from urllib.request import urlopen

DATA_URL = "https://raw.githubusercontent.com/hanisaf/python-programming-skills/main/data/"

with urlopen(DATA_URL + "InternetCompanies.csv") as response:
    lines = response.read().decode("utf-8").splitlines()

data = list(csv.DictReader(lines))
print(len(data), "records")
data[:3]

20 records


[{'Company': 'Apple',
  'Country': 'USA',
  'MarketCap': ' $547 ',
  'Cash': ' $233 ',
  'Revenue': ' $235 '},
 {'Company': 'Google/Alphabet',
  'Country': 'USA',
  'MarketCap': ' $510 ',
  'Cash': ' $79 ',
  'Revenue': ' $75 '},
 {'Company': 'Amazon',
  'Country': 'USA',
  'MarketCap': ' $341 ',
  'Cash': ' $16 ',
  'Revenue': ' $107 '}]

Why not just `split(",")` each line? Because real CSV files can contain commas *inside* quoted fields (for example `"Athens, GA"`). The `csv` module handles such cases correctly.

### 1.3 Reading JSON files

**JSON** (JavaScript Object Notation) is the most common format for data exchanged between web applications. JSON objects look just like Python dictionaries. Many data files store one JSON object per line (a format called *JSON Lines*). The `json` module converts each line to a dictionary:

In [2]:
import json

with urlopen(DATA_URL + "courses.json") as response:
    courses = [json.loads(line) for line in response.read().decode("utf-8").splitlines()]

with urlopen(DATA_URL + "instructors.json") as response:
    instructors = [json.loads(line) for line in response.read().decode("utf-8").splitlines()]

print(courses[0])
print(instructors[0])

{'cid': 0, 'number': '4600', 'major': 'MIST', 'title': 'Computer Programming in Business', 'currentInstructor': 4}
{'id': 0, 'name': 'Hani Safadi', 'courses': [0, 3]}


Notice that JSON values keep their types: `cid` is an integer, and the missing instructor of one course is `null` in JSON, which becomes `None` in Python.

The two files are related: each course stores the `id` of its current instructor. To combine them, build a dictionary that looks instructors up by `id`, then use it while looping over the courses. This is a **join**, the same operation that connects tables in a database.

In [3]:
instructor_by_id = {person["id"]: person["name"] for person in instructors}

for course in courses:
    name = instructor_by_id.get(course["currentInstructor"], "(not assigned)")
    print(f'{course["major"]} {course["number"]:<5} {course["title"]:<40} {name}')

MIST 4600  Computer Programming in Business         Karen Aguar
MIST 4610  Data Managment and Analytics             Nikhil Srinivasan
MIST 4630  Network-Based Application Development    Craig Piercy
MIST 5730  Advanced Data Management and Analytics   Hani Safadi
BUSN 3700  Principles of FinTech                    (not assigned)


### 1.4 The case: Internet companies

The file `InternetCompanies.csv` describes large technology and Internet companies in 2015. Market capitalization (`MarketCap`), `Cash`, and `Revenue` are in billions of US dollars. Private companies do not publish cash or revenue, so these values are missing and shown as `$-`.

We will answer the following questions.

**Part 1: Extract, transform, load (ETL)**

Convert `MarketCap`, `Cash`, and `Revenue` to numbers in dollars (multiply by one billion) and replace the missing values with `None`. Show the first and the last record.

**Part 2: Analytics**

1. What is the average market capitalization?
2. What is the average cash?
3. What is the average cash-to-revenue ratio (cash ÷ revenue)?
4. Which companies have a cash-to-revenue ratio below 0.5? (sorted by name)
5. How many companies do not report cash?
6. How many companies are there in each country?
7. Which company has a name that starts with a lower-case letter?
8. What is the total market capitalization per country?

Averages are rounded to two decimals.

## 2 · Warmup: Extract, transform, load

The raw values are strings with a dollar sign and extra spaces:

In [4]:
data[7]

{'Company': 'Uber',
 'Country': 'USA',
 'MarketCap': ' $63 ',
 'Cash': ' $-   ',
 'Revenue': ' $-   '}

**Exercise 2.1.** Write a function that converts one raw value such as `" $547 "` to a number of dollars, or to `None` when the value is missing.

In [5]:
BILLION = 1_000_000_000

def to_dollars(raw):
    """Convert a raw value such as ' $547 ' to dollars; return None for ' $-   '."""
    value = raw.strip().lstrip("$").strip()
    if value == "-":
        return None
    return int(value) * BILLION

print(to_dollars(" $547 "), to_dollars(" $-   "))

547000000000 None


Underscores in a number literal (`1_000_000_000`) are ignored by Python; they just make large numbers readable. Naming the constant `BILLION` documents what the multiplication means.

**Exercise 2.2.** Write a function that converts a raw record into a clean one, and apply it to every record.

In [6]:
def clean_record(record):
    """Return a copy of record with the money fields converted to dollars."""
    return {
        "Company": record["Company"],
        "Country": record["Country"],
        "MarketCap": to_dollars(record["MarketCap"]),
        "Cash": to_dollars(record["Cash"]),
        "Revenue": to_dollars(record["Revenue"]),
    }

companies = [clean_record(record) for record in data]
companies[0]

{'Company': 'Apple',
 'Country': 'USA',
 'MarketCap': 547000000000,
 'Cash': 233000000000,
 'Revenue': 235000000000}

In [7]:
companies[-1]

{'Company': 'Didi Kuaidi',
 'Country': 'China',
 'MarketCap': 25000000000,
 'Cash': None,
 'Revenue': None}

Breaking the cleaning into two small functions makes each one easy to test on a single value before applying it to the whole dataset.

## 3 · Workout: Analytics

Missing values need care: we cannot add `None` to a number. A helper that averages only the values that are present keeps the rest of the code simple.

In [8]:
def average(values):
    """Return the mean of the values that are not None, rounded to 2 decimals."""
    present = [v for v in values if v is not None]
    return round(sum(present) / len(present), 2)

**Questions 1–2.** Average market capitalization and average cash.

In [9]:
avg_market_cap = average([c["MarketCap"] for c in companies])
avg_cash = average([c["Cash"] for c in companies])

print(f"Average market cap: ${avg_market_cap:,.0f}")
print(f"Average cash:       ${avg_cash:,.0f}")

Average market cap: $138,250,000,000
Average cash:       $29,733,333,333


**Question 3.** Average cash-to-revenue ratio. The ratio exists only for companies that report both values.

In [10]:
ratios = {c["Company"]: c["Cash"] / c["Revenue"]
          for c in companies
          if c["Cash"] is not None and c["Revenue"] is not None}

average(ratios.values())

0.91

**Question 4.** Companies with a ratio below 0.5, sorted by name.

In [11]:
sorted(name for name, ratio in ratios.items() if ratio < 0.5)

['Amazon', 'JD.com', 'Netflix']

**Question 5.** Companies that do not report cash.

In [12]:
sum(1 for c in companies if c["Cash"] is None)

5

**Question 6.** Number of companies per country: the counting pattern from Module 9.

In [13]:
companies_per_country = {}
for c in companies:
    country = c["Country"]
    companies_per_country[country] = companies_per_country.get(country, 0) + 1
companies_per_country

{'USA': 12, 'China': 7, 'Japan': 1}

**Question 7.** The company whose name starts with a lower-case letter.

In [14]:
[c["Company"] for c in companies if c["Company"][0].islower()]

['eBay']

**Question 8.** Total market capitalization per country. The keys of `companies_per_country` give the list of countries.

In [15]:
market_cap_per_country = {
    country: sum(c["MarketCap"] for c in companies if c["Country"] == country)
    for country in companies_per_country
}
market_cap_per_country

{'USA': 2101000000000, 'China': 638000000000, 'Japan': 26000000000}

### Communicating the results

Numbers like `2101000000000` are hard to read. A short formatted report is much easier to share:

In [16]:
print(f"{'Country':<8}{'Companies':>10}{'Market cap ($B)':>17}{'Share':>8}")
print("-" * 43)
total_cap = sum(market_cap_per_country.values())
for country, cap in sorted(market_cap_per_country.items(), key=lambda item: item[1], reverse=True):
    print(f"{country:<8}{companies_per_country[country]:>10}{cap / BILLION:>17,.0f}{cap / total_cap:>8.1%}")

Country  Companies  Market cap ($B)   Share
-------------------------------------------
USA             12            2,101   76.0%
China            7              638   23.1%
Japan            1               26    0.9%


US companies account for most of the value, a pattern that a chart would make even clearer (Modules 14–15).

## Summary

- A list of dictionaries is a simple, flexible representation of tabular data.
- `csv.DictReader` and `json.loads` turn files into such lists; `urlopen` reads them from the web.
- Clean data with small, testable functions, then apply them to every record with a comprehension.
- Handle missing values explicitly (`None`) and exclude them from calculations.
- Counting and grouping are dictionary patterns; filtering and transforming are comprehension patterns.
- Doing all this by hand takes effort. Starting with Module 12, `pandas` will do most of it in a single line.